# Model Estimation — fixed specification, one model per cell

Estimates the **35 ordered probit models exactly as specified**, with no variable selection. The
variable list in each cell is taken from the FINAL MODEL block of `CARE_Dashboard_Models.xlsx`, so
each cell reproduces one published model.

Every cell is **fully self-contained** — its own imports, its own data load, its own `OrderedModel`
call, its own printout. Nothing is shared between cells except the optional `MODELS` dictionary used
by the summary section at the end. Run any cell on its own, in any order.

**Inputs:** `event_data/{heat,cold,powerout,earthquake,flooding}_data.csv`
**Runtime:** 1–5 s per cell (no stepwise search), about 2 minutes for the whole notebook.

### What each cell does
1. Loads that event's data file (already filtered to respondents exposed to the event).
2. Restricts to workers (`empsta < 3`) for **WFH** and **WFO** only.
3. Drops rows with missing values on any model variable.
4. Fits an ordered probit (no intercept — the thresholds absorb it).
5. **Converts the thresholds and asserts they are strictly increasing.** `statsmodels` stores every
   threshold after the first as a log-difference internally; `transform_threshold_params()` recovers
   the actual cutpoints, which are the R-comparable ones. The assertion fails loudly if monotonicity
   is ever violated.
6. Prints coefficients, standard errors, z, p, significance stars, and the thresholds.

All 35 models have been run this way: **every one converged and every threshold vector is strictly
increasing.**

---

## ⚠ Two variables differ from the published file: `ac` and `hispanic`

The specification lists `ac` and `hispanic`. Both are miscoded in the raw CSV, so the cells below use
the cleaned versions `ac_c` and `hispanic_c` (see `Codebook.xlsx` → Data Quality Notes):

- **`hispanic`** is raw survey coding `1 = Yes, 2 = No` — not a 0/1 dummy. Used raw, its coefficient
  carries the **opposite sign** to "is Hispanic". Sample size is unchanged; only the sign flips.
  Affects: `heat|Car`, `heat|Home`, `heat|WFH`, `cold|Usual`, `cold|WFO`, `earthquake|Transit`, `powerout|Transit`.
- **`ac`** uses `-9` as a skip-logic "question not shown" code. `-9` is a real integer, so it survives
  `dropna()` and gets fitted as if it were a genuine value. Setting it to missing instead **shrinks the
  sample** in three models, badly:

  | model | N with `ac_c` | N with raw `ac` | lost |
  |---|---|---|---|
  | `cold \| Usual` | 1,057 | 2,434 | 1,377 |
  | `earthquake \| Home` | 182 | 611 | 429 |
  | `flooding \| Transit` | 323 | 1,041 | 718 |

  In the Heat models `ac` costs only 1 observation, because the A/C question was routed to
  heat-exposed respondents.

**This needs your decision for those three models** — either drop `ac` from the specification, or keep
raw `ac` and accept that `-9` is in the regressor. Each affected cell has the raw alternative on a
commented line so you can switch and re-run. Everything else reproduces the published coefficients to
within 0.0005, which is just the 3-decimal rounding in the source file.

## Optional — collect results

Run this if you want the summary section at the end to build the combined CSV. Not needed to run any individual model cell.

In [5]:
MODELS = {}
print('MODELS ready')

MODELS ready


---

# Extreme Heat

`event_data/heat_data.csv` — 9 models.

### Extreme Heat — Go about business as usual  (`Usual`)

Dependent variable: `ext_heat_lkly_normal_business`  |  15 terms  |  N = 2,564

In [6]:
# Extreme Heat - Go about business as usual
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_lkly_normal_business'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'CR',
    'PR',
    'female',
    'age_3150',
    'age_5165',
    'non_wrkr',
    'dis_yes',
    'work_out',
    'hhveh0',
    'ac_c',
    'PopDens_medium',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['heat_imp_2', 'heat_imp_3', 'heat_imp_4', 'heat_imp_5', 'CR', 'PR', 'female', 'age_3150', 'age_5165', 'non_wrkr', 'dis_yes', 'work_out', 'hhveh0', 'ac_raw', 'PopDens_medium']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Go about business as usual   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Usual')] = tbl
except NameError: pass

Extreme Heat - Go about business as usual   dv=ext_heat_lkly_normal_business   (5 categories)
N = 2653   logL = -3775.61   AIC = 7589.23   pseudo-R2 = 0.0542
thresholds strictly increasing: True   ->  [-2.1531 -1.4518 -0.8482  0.1812]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
    heat_imp_2   -0.6594    0.0685   -9.6241    0.0000 ***
    heat_imp_3   -0.9079    0.0685  -13.2522    0.0000 ***
    heat_imp_4   -1.0891    0.0820  -13.2891    0.0000 ***
    heat_imp_5   -1.0324    0.0952  -10.8396    0.0000 ***
            CR    0.0817    0.0262    3.1145    0.0018  **
            PR    0.1821    0.0266    6.8374    0.0000 ***
        female   -0.1110    0.0439   -2.5248    0.0116   *
      age_3150    0.1879    0.0519    3.6175    0.0003 ***
      age_5165    0.1095    0.0540    2.0272    0.0426   *
      non_wrkr   -0.2056    0.0469   -4.3858    0.0000 ***
       dis_yes   -0.1276    0.0535   

### Extreme Heat — Staying at home  (`Home`)

Dependent variable: `ext_heat_stay_home`  |  14 terms  |  N = 2,564

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [7]:
# Extreme Heat - Staying at home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_stay_home'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'CR',
    'non_wrkr',
    'edu_bs',
    'hispanic_c',
    'hhsize2',
    'dis_yes',
    'work_out',
    'tcom_no',
    'ac_c',
    'PopDens_high',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['heat_imp_2', 'heat_imp_3', 'heat_imp_4', 'heat_imp_5', 'CR', 'non_wrkr', 'edu_bs', 'hispanic_c', 'hhsize2', 'dis_yes', 'work_out', 'tcom_no', 'ac_raw', 'PopDens_high']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Staying at home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Home')] = tbl
except NameError: pass

Extreme Heat - Staying at home   dv=ext_heat_stay_home   (3 categories)
N = 2653   logL = -2293.72   AIC = 4619.44   pseudo-R2 = 0.0190
thresholds strictly increasing: True   ->  [-0.9555  0.7635]
------------------------------------------------------------------------------
    Variable      Coef        SE         z         p sig
  heat_imp_2    0.2721    0.0707    3.8490    0.0001 ***
  heat_imp_3    0.2910    0.0704    4.1336    0.0000 ***
  heat_imp_4    0.4787    0.0868    5.5147    0.0000 ***
  heat_imp_5    0.3867    0.1016    3.8076    0.0001 ***
          CR    0.0860    0.0278    3.0933    0.0020  **
    non_wrkr    0.1590    0.0572    2.7805    0.0054  **
      edu_bs    0.1037    0.0549    1.8902    0.0587   +
  hispanic_c    0.1586    0.0871    1.8211    0.0686   +
     hhsize2   -0.0844    0.0484   -1.7419    0.0815   +
     dis_yes    0.1471    0.0592    2.4848    0.0130   *
    work_out   -0.2374    0.1128   -2.1047    0.0353   *
     tcom_no    0.1238    0.0645    1.92

### Extreme Heat — Using a car for traveling  (`Car`)

Dependent variable: `ext_heat_car_travel`  |  13 terms  |  N = 2,564

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [8]:
# Extreme Heat - Using a car for traveling
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_car_travel'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'SE',
    'PR',
    'non_wrkr',
    'white',
    'black',
    'hispanic_c',
    'hhsize2',
    'sa_home',
    'PopDens_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Using a car for traveling   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Car')] = tbl
except NameError: pass

Extreme Heat - Using a car for traveling   dv=ext_heat_car_travel   (3 categories)
N = 2653   logL = -2348.15   AIC = 4726.29   pseudo-R2 = 0.0169
thresholds strictly increasing: True   ->  [-0.7701  1.1054]
------------------------------------------------------------------------------
    Variable      Coef        SE         z         p sig
  heat_imp_2    0.0531    0.0709    0.7490    0.4539    
  heat_imp_3    0.1276    0.0707    1.8060    0.0709   +
  heat_imp_4    0.1569    0.0846    1.8537    0.0638   +
  heat_imp_5    0.1674    0.0987    1.6965    0.0898   +
          SE    0.0763    0.0280    2.7239    0.0065  **
          PR    0.0748    0.0281    2.6628    0.0077  **
    non_wrkr   -0.1600    0.0474   -3.3772    0.0007 ***
       white    0.1738    0.0895    1.9414    0.0522   +
       black    0.2464    0.1027    2.3988    0.0165   *
  hispanic_c    0.1558    0.0858    1.8166    0.0693   +
     hhsize2   -0.1153    0.0481   -2.3987    0.0165   *
     sa_home   -0.0827    0.0

### Extreme Heat — Taking public transit  (`Transit`)

Dependent variable: `ext_heat_public_transit`  |  13 terms  |  N = 2,565

In [9]:
# Extreme Heat - Taking public transit
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_public_transit'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'PR',
    'age_3150',
    'age_5165',
    'age_65p',
    'in50100',
    'work_out',
    'tcom_no',
    'hhveh0',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Taking public transit   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Transit')] = tbl
except NameError: pass

Extreme Heat - Taking public transit   dv=ext_heat_public_transit   (3 categories)
N = 2653   logL = -2138.55   AIC = 4307.10   pseudo-R2 = 0.0127
thresholds strictly increasing: True   ->  [-0.9417  1.1699]
------------------------------------------------------------------------------
          Variable      Coef        SE         z         p sig
        heat_imp_2   -0.0889    0.0732   -1.2142    0.2247    
        heat_imp_3   -0.1408    0.0728   -1.9322    0.0533   +
        heat_imp_4   -0.1063    0.0877   -1.2127    0.2253    
        heat_imp_5   -0.1457    0.1014   -1.4365    0.1509    
                PR   -0.0595    0.0287   -2.0707    0.0384   *
          age_3150   -0.1563    0.0753   -2.0757    0.0379   *
          age_5165   -0.1534    0.0799   -1.9209    0.0547   +
           age_65p   -0.1571    0.0801   -1.9610    0.0499   *
           in50100    0.1133    0.0508    2.2283    0.0259   *
          work_out    0.4651    0.1134    4.1019    0.0000 ***
           tcom_no  

### Extreme Heat — Working from home  (`WFH`)

Dependent variable: `ext_heat_wfh`  |  14 terms  |  N = 1,213  |  workers only (`empsta < 3`)

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [48]:
# Extreme Heat - Working from home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_heat_wfh'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'CR',
    'edu_bs',
    'black',
    'hispanic_c',
    'in50',
    'in50100',
    'hhsize2',
    'tcom_no',
    'ac_c',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['heat_imp_2', 'heat_imp_3', 'heat_imp_4', 'heat_imp_5', 'CR', 'edu_bs', 'black', 'hispanic_c', 'in50', 'in50100', 'hhsize2', 'tcom_no', 'ac_raw', 'NatWalkInd_high']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Working from home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'WFH')] = tbl
except NameError: pass

Extreme Heat - Working from home   dv=ext_heat_wfh   (3 categories)
N = 1271   logL = -980.11   AIC = 1990.23   pseudo-R2 = 0.0534
thresholds strictly increasing: True   ->  [-1.5422  0.6616]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 heat_imp_2    0.0007    0.1145    0.0059    0.9953    
 heat_imp_3    0.0265    0.1115    0.2373    0.8124    
 heat_imp_4    0.3355    0.1276    2.6284    0.0086  **
 heat_imp_5    0.0680    0.1474    0.4613    0.6446    
         CR    0.1208    0.0434    2.7854    0.0053  **
     edu_bs    0.1551    0.0776    1.9984    0.0457   *
      black    0.2157    0.0888    2.4289    0.0151   *
 hispanic_c    0.2884    0.1109    2.6003    0.0093  **
       in50   -0.1772    0.0888   -1.9950    0.0460   *
    in50100   -0.1500    0.0880   -1.7043    0.0883   +
    hhsize2   -0.1721    0.0757   -2.2732    0.0230   *
    tcom_no   -0.4332    0.0696   -6.2256    0.0000 ***
 

### Extreme Heat — Working from the office  (`WFO`)

Dependent variable: `ext_heat_commute`  |  10 terms  |  N = 1,271  |  workers only (`empsta < 3`)

In [49]:
# Extreme Heat - Working from the office
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_heat_commute'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'CR',
    'edu_bs',
    'in50',
    'hhsize1',
    'work_out',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Working from the office   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'WFO')] = tbl
except NameError: pass

Extreme Heat - Working from the office   dv=ext_heat_commute   (3 categories)
N = 1271   logL = -1088.31   AIC = 2200.61   pseudo-R2 = 0.0181
thresholds strictly increasing: True   ->  [-0.8253  1.1341]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 heat_imp_2   -0.0305    0.1106   -0.2760    0.7826    
 heat_imp_3   -0.0343    0.1076   -0.3190    0.7497    
 heat_imp_4   -0.0833    0.1235   -0.6746    0.5000    
 heat_imp_5    0.3184    0.1424    2.2353    0.0254   *
         CR   -0.0830    0.0415   -1.9975    0.0458   *
     edu_bs   -0.2044    0.0748   -2.7335    0.0063  **
       in50   -0.1170    0.0707   -1.6557    0.0978   +
    hhsize1    0.1794    0.0792    2.2652    0.0235   *
   work_out    0.2173    0.1139    1.9085    0.0563   +
    tcom_no    0.1804    0.0663    2.7226    0.0065  **
threshold_1   -0.8253       NaN       NaN       NaN    
threshold_2    1.1341       NaN       NaN     

### Extreme Heat — Eating indoors at a restaurant  (`Dine in`)

Dependent variable: `ext_heat_indoor_restaurant`  |  8 terms  |  N = 2,565

In [50]:
# Extreme Heat - Eating indoors at a restaurant
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_indoor_restaurant'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'non_wrkr',
    'edu_bs',
    'hhsize2',
    'Diversity_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Eating indoors at a restaurant   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Dine in')] = tbl
except NameError: pass

Extreme Heat - Eating indoors at a restaurant   dv=ext_heat_indoor_restaurant   (3 categories)
N = 2653   logL = -2449.61   AIC = 4919.22   pseudo-R2 = 0.0126
thresholds strictly increasing: True   ->  [-0.9566  0.798 ]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
    heat_imp_2    0.1011    0.0698    1.4491    0.1473    
    heat_imp_3    0.0054    0.0692    0.0786    0.9373    
    heat_imp_4   -0.0915    0.0832   -1.1004    0.2711    
    heat_imp_5    0.2590    0.0970    2.6699    0.0076  **
      non_wrkr   -0.2318    0.0452   -5.1314    0.0000 ***
        edu_bs   -0.1247    0.0531   -2.3489    0.0188   *
       hhsize2   -0.1123    0.0473   -2.3734    0.0176   *
Diversity_high   -0.1365    0.0477   -2.8614    0.0042  **
   threshold_1   -0.9566       NaN       NaN       NaN    
   threshold_2    0.7980       NaN       NaN       NaN    
----------------------------------------------------

### Extreme Heat — Picking up takeout  (`Pick up`)

Dependent variable: `ext_heat_takeout_pickup`  |  12 terms  |  N = 2,564

In [51]:
# Extreme Heat - Picking up takeout
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_takeout_pickup'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'SE',
    'PR',
    'age_5165',
    'age_65p',
    'hhveh0',
    'ac_c',
    'EmpDens_medium',
    'Diversity_high',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['heat_imp_2', 'heat_imp_3', 'heat_imp_4', 'heat_imp_5', 'SE', 'PR', 'age_5165', 'age_65p', 'hhveh0', 'ac_raw', 'EmpDens_medium', 'Diversity_high']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Picking up takeout   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Pick up')] = tbl
except NameError: pass

Extreme Heat - Picking up takeout   dv=ext_heat_takeout_pickup   (3 categories)
N = 2653   logL = -2452.78   AIC = 4933.57   pseudo-R2 = 0.0118
thresholds strictly increasing: True   ->  [-1.0492  0.7021]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
    heat_imp_2   -0.0490    0.0700   -0.6990    0.4846    
    heat_imp_3    0.0168    0.0699    0.2403    0.8101    
    heat_imp_4    0.0544    0.0841    0.6469    0.5177    
    heat_imp_5    0.0296    0.0978    0.3031    0.7618    
            SE    0.0817    0.0275    2.9701    0.0030  **
            PR    0.0495    0.0277    1.7877    0.0738   +
      age_5165   -0.1502    0.0567   -2.6483    0.0081  **
       age_65p   -0.1794    0.0539   -3.3277    0.0009 ***
        hhveh0   -0.1380    0.0702   -1.9649    0.0494   *
          ac_c   -0.1233    0.0652   -1.8915    0.0586   +
EmpDens_medium   -0.0987    0.0473   -2.0863    0.0370   *
Diversit

### Extreme Heat — Having food delivered  (`Delivery`)

Dependent variable: `ext_heat_food_delivery`  |  15 terms  |  N = 2,564

In [52]:
# Extreme Heat - Having food delivered
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/heat_data.csv', low_memory=False)
dv = 'ext_heat_food_delivery'

x_vars = [
    'heat_imp_2',
    'heat_imp_3',
    'heat_imp_4',
    'heat_imp_5',
    'CR',
    'PR',
    'age_65p',
    'non_wrkr',
    'hhsize2',
    'dis_yes',
    'hhveh0',
    'ac_c',
    'rural',
    'NetworkDensity_medium',
    'TransitAccess_high',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['heat_imp_2', 'heat_imp_3', 'heat_imp_4', 'heat_imp_5', 'CR', 'PR', 'age_65p', 'non_wrkr', 'hhsize2', 'dis_yes', 'hhveh0', 'ac_raw', 'rural', 'NetworkDensity_medium', 'TransitAccess_high']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Heat - Having food delivered   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('heat', 'Delivery')] = tbl
except NameError: pass

Extreme Heat - Having food delivered   dv=ext_heat_food_delivery   (3 categories)
N = 2653   logL = -2307.76   AIC = 4649.53   pseudo-R2 = 0.0323
thresholds strictly increasing: True   ->  [-1.0017  0.9012]
------------------------------------------------------------------------------
             Variable      Coef        SE         z         p sig
           heat_imp_2    0.1558    0.0713    2.1840    0.0290   *
           heat_imp_3    0.1459    0.0712    2.0503    0.0403   *
           heat_imp_4    0.3361    0.0860    3.9085    0.0001 ***
           heat_imp_5    0.4356    0.1010    4.3125    0.0000 ***
                   CR    0.0674    0.0285    2.3677    0.0179   *
                   PR    0.0699    0.0286    2.4466    0.0144   *
              age_65p   -0.1694    0.0585   -2.8937    0.0038  **
             non_wrkr   -0.2631    0.0532   -4.9438    0.0000 ***
              hhsize2   -0.0884    0.0489   -1.8085    0.0705   +
              dis_yes    0.0818    0.0584    1.3993   

---

# Extreme Cold

`event_data/cold_data.csv` — 9 models.

### Extreme Cold — Go about business as usual  (`Usual`)

Dependent variable: `ext_cold_lkly_normal_business`  |  13 terms  |  N = 1,057

> ⚠ **`ac_c` cuts this sample from 2,434 to 1,057.** The published model used raw `ac` (with `-9` treated as a value). Swap the commented line below to reproduce it.

In [53]:
# Extreme Cold - Go about business as usual
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_lkly_normal_business'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'CR',
    'PR',
    'female',
    'age_3150',
    'non_wrkr',
    'white',
    'sa_home',
    'ac_c',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['cold_imp_2', 'cold_imp_3', 'cold_imp_4', 'cold_imp_5', 'CR', 'PR', 'female', 'age_3150', 'non_wrkr', 'white', 'hispanic_c', 'sa_home', 'ac_raw']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Go about business as usual   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Usual')] = tbl
except NameError: pass

Extreme Cold - Go about business as usual   dv=ext_cold_lkly_normal_business   (5 categories)
N = 2434   logL = -3451.47   AIC = 6934.94   pseudo-R2 = 0.0453
thresholds strictly increasing: True   ->  [-2.2827 -1.4744 -0.831   0.178 ]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 cold_imp_2   -0.5844    0.0658   -8.8847    0.0000 ***
 cold_imp_3   -0.7999    0.0674  -11.8705    0.0000 ***
 cold_imp_4   -0.9093    0.0803  -11.3183    0.0000 ***
 cold_imp_5   -0.9705    0.1019   -9.5200    0.0000 ***
         CR    0.0894    0.0263    3.4025    0.0007 ***
         PR    0.1944    0.0275    7.0633    0.0000 ***
     female   -0.1387    0.0448   -3.0978    0.0019  **
   age_3150    0.1010    0.0484    2.0876    0.0368   *
   non_wrkr   -0.2239    0.0474   -4.7269    0.0000 ***
      white    0.2060    0.0554    3.7205    0.0002 ***
    sa_home   -0.0802    0.0443   -1.8118    0.0700   +
       ac_c   

### Extreme Cold — Staying at home  (`Home`)

Dependent variable: `ext_cold_stay_home`  |  8 terms  |  N = 2,434

In [54]:
# Extreme Cold - Staying at home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_stay_home'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'CR',
    'SE',
    'dis_yes',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Staying at home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Home')] = tbl
except NameError: pass

Extreme Cold - Staying at home   dv=ext_cold_stay_home   (3 categories)
N = 2434   logL = -1991.62   AIC = 4003.23   pseudo-R2 = 0.0150
thresholds strictly increasing: True   ->  [-1.457   0.2142]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 cold_imp_2    0.2976    0.0699    4.2570    0.0000 ***
 cold_imp_3    0.4575    0.0723    6.3258    0.0000 ***
 cold_imp_4    0.3336    0.0858    3.8870    0.0001 ***
 cold_imp_5    0.4300    0.1120    3.8406    0.0001 ***
         CR    0.0762    0.0288    2.6472    0.0081  **
         SE   -0.0793    0.0300   -2.6432    0.0082  **
    dis_yes    0.1125    0.0639    1.7622    0.0780   +
    tcom_no   -0.0988    0.0568   -1.7404    0.0818   +
threshold_1   -1.4570       NaN       NaN       NaN    
threshold_2    0.2142       NaN       NaN       NaN    
------------------------------------------------------------------------------ 
sig: *** p<0.001  ** p<0.01 

### Extreme Cold — Using a car for traveling  (`Car`)

Dependent variable: `ext_cold_car_travel`  |  13 terms  |  N = 2,374

In [55]:
# Extreme Cold - Using a car for traveling
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_car_travel'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'SE',
    'female',
    'age_5165',
    'age_65p',
    'non_wrkr',
    'black',
    'in50100',
    'sa_home',
    'PopDens_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Using a car for traveling   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Car')] = tbl
except NameError: pass

Extreme Cold - Using a car for traveling   dv=ext_cold_car_travel   (3 categories)
N = 2434   logL = -2301.78   AIC = 4633.57   pseudo-R2 = 0.0241
thresholds strictly increasing: True   ->  [-0.7447  0.8815]
------------------------------------------------------------------------------
    Variable      Coef        SE         z         p sig
  cold_imp_2   -0.0595    0.0678   -0.8781    0.3799    
  cold_imp_3   -0.0560    0.0697   -0.8040    0.4214    
  cold_imp_4    0.0213    0.0840    0.2533    0.8000    
  cold_imp_5    0.1879    0.1071    1.7544    0.0794   +
          SE    0.0823    0.0289    2.8481    0.0044  **
      female   -0.0821    0.0488   -1.6825    0.0925   +
    age_5165   -0.1309    0.0615   -2.1291    0.0332   *
     age_65p   -0.1486    0.0665   -2.2354    0.0254   *
    non_wrkr   -0.1253    0.0558   -2.2440    0.0248   *
       black    0.2082    0.0693    3.0050    0.0027  **
     in50100   -0.0969    0.0516   -1.8770    0.0605   +
     sa_home   -0.1087    0.0

### Extreme Cold — Taking public transit  (`Transit`)

Dependent variable: `ext_cold_transit_use`  |  14 terms  |  N = 2,374

In [56]:
# Extreme Cold - Taking public transit
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_transit_use'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'CR',
    'asian',
    'in50',
    'in50100',
    'hhsize2',
    'child',
    'work_out',
    'sa_home',
    'hhveh0',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Taking public transit   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Transit')] = tbl
except NameError: pass

Extreme Cold - Taking public transit   dv=ext_cold_transit_use   (3 categories)
N = 2434   logL = -2038.03   AIC = 4108.06   pseudo-R2 = 0.0166
thresholds strictly increasing: True   ->  [-1.0464  0.959 ]
------------------------------------------------------------------------------
          Variable      Coef        SE         z         p sig
        cold_imp_2   -0.0805    0.0704   -1.1440    0.2526    
        cold_imp_3   -0.1697    0.0717   -2.3650    0.0180   *
        cold_imp_4   -0.1870    0.0861   -2.1722    0.0298   *
        cold_imp_5    0.1287    0.1087    1.1839    0.2365    
                CR    0.0886    0.0283    3.1279    0.0018  **
             asian   -0.2443    0.1411   -1.7315    0.0834   +
              in50   -0.1875    0.0664   -2.8251    0.0047  **
           in50100   -0.1888    0.0689   -2.7388    0.0062  **
           hhsize2   -0.1072    0.0535   -2.0036    0.0451   *
             child   -0.1739    0.0595   -2.9249    0.0034  **
          work_out    0

### Extreme Cold — Working from home  (`WFH`)

Dependent variable: `ext_cold_wfh`  |  12 terms  |  N = 1,179  |  workers only (`empsta < 3`)

In [57]:
# Extreme Cold - Working from home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_cold_wfh'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'female',
    'bs_grad',
    'white',
    'black',
    'asian',
    'in50100',
    'child',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Working from home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'WFH')] = tbl
except NameError: pass

Extreme Cold - Working from home   dv=ext_cold_wfh   (3 categories)
N = 1179   logL = -968.04   AIC = 1964.08   pseudo-R2 = 0.0493
thresholds strictly increasing: True   ->  [-1.9137  0.0785]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 cold_imp_2    0.0531    0.1113    0.4775    0.6330    
 cold_imp_3    0.2411    0.1116    2.1605    0.0307   *
 cold_imp_4    0.0361    0.1255    0.2879    0.7735    
 cold_imp_5    0.2028    0.1547    1.3109    0.1899    
     female   -0.1497    0.0697   -2.1470    0.0318   *
    bs_grad    0.2033    0.0726    2.8018    0.0051  **
      white   -0.4141    0.1510   -2.7416    0.0061  **
      black   -0.3208    0.1624   -1.9750    0.0483   *
      asian   -0.4984    0.2132   -2.3372    0.0194   *
    in50100    0.1158    0.0728    1.5900    0.1118    
      child    0.2054    0.0726    2.8278    0.0047  **
    tcom_no   -0.4524    0.0718   -6.2959    0.0000 ***
t

### Extreme Cold — Working from the office  (`WFO`)

Dependent variable: `ext_cold_commute`  |  11 terms  |  N = 1,179  |  workers only (`empsta < 3`)

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [58]:
# Extreme Cold - Working from the office
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_cold_commute'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'age_3150',
    'bs_grad',
    'white',
    'black',
    'asian',
    'hispanic_c',
    'work_out',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Working from the office   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'WFO')] = tbl
except NameError: pass

Extreme Cold - Working from the office   dv=ext_cold_commute   (3 categories)
N = 1179   logL = -1082.34   AIC = 2190.67   pseudo-R2 = 0.0222
thresholds strictly increasing: True   ->  [-0.3092  1.4285]
------------------------------------------------------------------------------
   Variable      Coef        SE         z         p sig
 cold_imp_2   -0.0451    0.1084   -0.4166    0.6770    
 cold_imp_3   -0.1762    0.1076   -1.6370    0.1016    
 cold_imp_4   -0.0852    0.1213   -0.7026    0.4823    
 cold_imp_5    0.2688    0.1493    1.8004    0.0718   +
   age_3150   -0.1364    0.0670   -2.0341    0.0419   *
    bs_grad   -0.1999    0.0682   -2.9330    0.0034  **
      white    0.4541    0.1507    3.0131    0.0026  **
      black    0.5737    0.1622    3.5361    0.0004 ***
      asian    0.3777    0.2099    1.7994    0.0720   +
 hispanic_c    0.2442    0.1106    2.2072    0.0273   *
   work_out    0.2761    0.1186    2.3275    0.0199   *
threshold_1   -0.3092       NaN       NaN     

### Extreme Cold — Eating indoors at a restaurant  (`Dine in`)

Dependent variable: `ext_cold_indoor_restaurant`  |  11 terms  |  N = 2,374

In [59]:
# Extreme Cold - Eating indoors at a restaurant
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_indoor_restaurant'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'SE',
    'female',
    'age_3150',
    'non_wrkr',
    'in50100',
    'PopDens_high',
    'NatWalkInd_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Eating indoors at a restaurant   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Dine in')] = tbl
except NameError: pass

Extreme Cold - Eating indoors at a restaurant   dv=ext_cold_indoor_restaurant   (3 categories)
N = 2434   logL = -2305.03   AIC = 4636.07   pseudo-R2 = 0.0205
thresholds strictly increasing: True   ->  [-0.5987  0.9511]
------------------------------------------------------------------------------
       Variable      Coef        SE         z         p sig
     cold_imp_2   -0.1375    0.0677   -2.0300    0.0424   *
     cold_imp_3   -0.2249    0.0695   -3.2346    0.0012  **
     cold_imp_4   -0.1511    0.0836   -1.8081    0.0706   +
     cold_imp_5    0.1282    0.1060    1.2088    0.2267    
             SE    0.0859    0.0289    2.9731    0.0029  **
         female   -0.1233    0.0482   -2.5567    0.0106   *
       age_3150    0.1035    0.0513    2.0158    0.0438   *
       non_wrkr   -0.1669    0.0513   -3.2538    0.0011  **
        in50100   -0.2274    0.0516   -4.4034    0.0000 ***
   PopDens_high    0.0954    0.0506    1.8855    0.0594   +
NatWalkInd_high    0.1507    0.0572    2.

### Extreme Cold — Picking up takeout  (`Pick up`)

Dependent variable: `ext_cold_takeout_pickup`  |  10 terms  |  N = 2,374

In [60]:
# Extreme Cold - Picking up takeout
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_takeout_pickup'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'SE',
    'PR',
    'female',
    'non_wrkr',
    'in50100',
    'PopDens_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Picking up takeout   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Pick up')] = tbl
except NameError: pass

Extreme Cold - Picking up takeout   dv=ext_cold_takeout_pickup   (3 categories)
N = 2434   logL = -2363.61   AIC = 4751.22   pseudo-R2 = 0.0155
thresholds strictly increasing: True   ->  [-0.6332  0.875 ]
------------------------------------------------------------------------------
    Variable      Coef        SE         z         p sig
  cold_imp_2   -0.0151    0.0673   -0.2241    0.8227    
  cold_imp_3   -0.0989    0.0690   -1.4326    0.1520    
  cold_imp_4   -0.0453    0.0830   -0.5462    0.5849    
  cold_imp_5    0.1051    0.1055    0.9959    0.3193    
          SE    0.0802    0.0287    2.7953    0.0052  **
          PR    0.0908    0.0289    3.1443    0.0017  **
      female   -0.1581    0.0480   -3.2949    0.0010 ***
    non_wrkr   -0.1396    0.0489   -2.8565    0.0043  **
     in50100   -0.1655    0.0511   -3.2420    0.0012  **
PopDens_high    0.0925    0.0501    1.8448    0.0651   +
 threshold_1   -0.6332       NaN       NaN       NaN    
 threshold_2    0.8750       NaN

### Extreme Cold — Having food delivered  (`Delivery`)

Dependent variable: `ext_cold_food_delivery`  |  13 terms  |  N = 2,374

In [61]:
# Extreme Cold - Having food delivered
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/cold_data.csv', low_memory=False)
dv = 'ext_cold_food_delivery'

x_vars = [
    'cold_imp_2',
    'cold_imp_3',
    'cold_imp_4',
    'cold_imp_5',
    'CR',
    'PR',
    'age_65p',
    'non_wrkr',
    'hhsize1',
    'hhsize2',
    'PopDens_high',
    'NetworkDensity_high',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Extreme Cold - Having food delivered   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('cold', 'Delivery')] = tbl
except NameError: pass

Extreme Cold - Having food delivered   dv=ext_cold_food_delivery   (3 categories)
N = 2434   logL = -2351.62   AIC = 4733.24   pseudo-R2 = 0.0239
thresholds strictly increasing: True   ->  [-1.0469  0.5457]
------------------------------------------------------------------------------
           Variable      Coef        SE         z         p sig
         cold_imp_2    0.0093    0.0673    0.1374    0.8907    
         cold_imp_3   -0.0187    0.0690   -0.2714    0.7861    
         cold_imp_4    0.0189    0.0830    0.2276    0.8200    
         cold_imp_5    0.2644    0.1070    2.4718    0.0134   *
                 CR    0.0690    0.0278    2.4817    0.0131   *
                 PR    0.0913    0.0290    3.1460    0.0017  **
            age_65p   -0.1363    0.0611   -2.2332    0.0255   *
           non_wrkr   -0.2045    0.0524   -3.9023    0.0001 ***
            hhsize1   -0.1176    0.0602   -1.9521    0.0509   +
            hhsize2   -0.1807    0.0570   -3.1709    0.0015  **
       Pop

---

# Power Outage

`event_data/powerout_data.csv` — 7 models.

### Power Outage — Go about business as usual  (`Usual`)

Dependent variable: `ext_powerout_lkly_normal_business`  |  15 terms  |  N = 2,300

In [62]:
# Power Outage - Go about business as usual
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_lkly_normal_business'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'CR',
    'SE',
    'PR',
    'female',
    'age_5165',
    'age_65p',
    'in50',
    'in50100',
    'work_out',
    'rural',
    'PopDens_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Go about business as usual   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Usual')] = tbl
except NameError: pass

Power Outage - Go about business as usual   dv=ext_powerout_lkly_normal_business   (5 categories)
N = 2381   logL = -3570.94   AIC = 7179.89   pseudo-R2 = 0.0324
thresholds strictly increasing: True   ->  [-2.0126 -1.2474 -0.6178  0.351 ]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
powerout_imp_2   -0.4187    0.0600   -6.9749    0.0000 ***
powerout_imp_3   -0.5379    0.0618   -8.7013    0.0000 ***
powerout_imp_4   -0.7678    0.0791   -9.7031    0.0000 ***
powerout_imp_5   -0.9175    0.1004   -9.1350    0.0000 ***
            CR    0.0838    0.0269    3.1193    0.0018  **
            SE    0.0992    0.0281    3.5345    0.0004 ***
            PR    0.0873    0.0278    3.1405    0.0017  **
        female   -0.1186    0.0460   -2.5779    0.0099  **
      age_5165   -0.1215    0.0558   -2.1787    0.0294   *
       age_65p   -0.3072    0.0540   -5.6925    0.0000 ***
          in50   -0.1329    0.057

### Power Outage — Staying at home  (`Home`)

Dependent variable: `ext_powerout_stay_home`  |  11 terms  |  N = 2,300

In [63]:
# Power Outage - Staying at home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_stay_home'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'PR',
    'age_5165',
    'age_65p',
    'black',
    'in50100',
    'dis_yes',
    'EmpDens_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Staying at home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Home')] = tbl
except NameError: pass

Power Outage - Staying at home   dv=ext_powerout_stay_home   (3 categories)
N = 2381   logL = -2271.69   AIC = 4569.38   pseudo-R2 = 0.0113
thresholds strictly increasing: True   ->  [-0.8943  0.6455]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
powerout_imp_2    0.1487    0.0642    2.3163    0.0205   *
powerout_imp_3    0.1694    0.0659    2.5695    0.0102   *
powerout_imp_4    0.0977    0.0835    1.1707    0.2417    
powerout_imp_5    0.2682    0.1060    2.5306    0.0114   *
            PR    0.0756    0.0294    2.5701    0.0102   *
      age_5165    0.1839    0.0596    3.0853    0.0020  **
       age_65p    0.2430    0.0574    4.2329    0.0000 ***
         black    0.1604    0.0708    2.2655    0.0235   *
       in50100   -0.0971    0.0507   -1.9156    0.0554   +
       dis_yes    0.1602    0.0616    2.5995    0.0093  **
  EmpDens_high    0.0923    0.0509    1.8145    0.0696   +
   threshold

### Power Outage — Using a car for traveling  (`Car`)

Dependent variable: `ext_powerout_car_travel`  |  10 terms  |  N = 2,381

In [64]:
# Power Outage - Using a car for traveling
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_car_travel'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'female',
    'age_5165',
    'age_65p',
    'child',
    'hhveh0',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Using a car for traveling   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Car')] = tbl
except NameError: pass

Power Outage - Using a car for traveling   dv=ext_powerout_car_travel   (3 categories)
N = 2381   logL = -2174.31   AIC = 4370.62   pseudo-R2 = 0.0217
thresholds strictly increasing: True   ->  [-1.15    0.6312]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
powerout_imp_2   -0.0109    0.0646   -0.1694    0.8655    
powerout_imp_3    0.0662    0.0663    0.9982    0.3182    
powerout_imp_4    0.0887    0.0841    1.0547    0.2916    
powerout_imp_5    0.1959    0.1056    1.8564    0.0634   +
        female   -0.1702    0.0484   -3.5201    0.0004 ***
      age_5165   -0.2286    0.0635   -3.6033    0.0003 ***
       age_65p   -0.2662    0.0626   -4.2505    0.0000 ***
         child    0.1279    0.0614    2.0819    0.0374   *
        hhveh0   -0.3210    0.0731   -4.3898    0.0000 ***
   threshold_1   -1.1500       NaN       NaN       NaN    
   threshold_2    0.6312       NaN       NaN       NaN    
-

### Power Outage — Taking public transit  (`Transit`)

Dependent variable: `ext_powerout_transit_use`  |  17 terms  |  N = 2,380

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [65]:
# Power Outage - Taking public transit
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_transit_use'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'CR',
    'SE',
    'PR',
    'female',
    'age_3150',
    'age_5165',
    'age_65p',
    'hispanic_c',
    'in50',
    'in50100',
    'child',
    'sa_home',
    'hhveh0',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Taking public transit   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Transit')] = tbl
except NameError: pass

Power Outage - Taking public transit   dv=ext_powerout_transit_use   (3 categories)
N = 2381   logL = -1913.54   AIC = 3865.07   pseudo-R2 = 0.0237
thresholds strictly increasing: True   ->  [-1.2152  0.9101]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
powerout_imp_2   -0.0879    0.0674   -1.3046    0.1920    
powerout_imp_3   -0.0541    0.0692   -0.7820    0.4342    
powerout_imp_4   -0.0146    0.0874   -0.1673    0.8671    
powerout_imp_5    0.1558    0.1088    1.4317    0.1522    
            CR    0.0685    0.0302    2.2716    0.0231   *
            SE    0.0568    0.0317    1.7914    0.0732   +
            PR   -0.0529    0.0313   -1.6917    0.0907   +
        female   -0.0979    0.0517   -1.8953    0.0581   +
      age_3150   -0.2185    0.0822   -2.6576    0.0079  **
      age_5165   -0.2647    0.0893   -2.9655    0.0030  **
       age_65p   -0.2715    0.0893   -3.0392    0.0024  **
    

### Power Outage — Eating indoors at a restaurant  (`Dine in`)

Dependent variable: `ext_powerout_indoor_restaurant`  |  14 terms  |  N = 2,300

In [66]:
# Power Outage - Eating indoors at a restaurant
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_indoor_restaurant'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'SE',
    'PR',
    'female',
    'age_5165',
    'white',
    'in50',
    'in50100',
    'hhsize2',
    'NetworkDensity_medium',
    'NetworkDensity_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Eating indoors at a restaurant   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Dine in')] = tbl
except NameError: pass

Power Outage - Eating indoors at a restaurant   dv=ext_powerout_indoor_restaurant   (3 categories)
N = 2381   logL = -2353.54   AIC = 4739.09   pseudo-R2 = 0.0149
thresholds strictly increasing: True   ->  [-0.6192  0.8708]
------------------------------------------------------------------------------
             Variable      Coef        SE         z         p sig
       powerout_imp_2    0.0163    0.0636    0.2564    0.7977    
       powerout_imp_3    0.1178    0.0651    1.8102    0.0703   +
       powerout_imp_4    0.2706    0.0822    3.2934    0.0010 ***
       powerout_imp_5    0.2549    0.1034    2.4640    0.0137   *
                   SE    0.0560    0.0295    1.8954    0.0580   +
                   PR   -0.0576    0.0294   -1.9614    0.0498   *
               female   -0.1167    0.0486   -2.4027    0.0163   *
             age_5165   -0.0964    0.0544   -1.7725    0.0763   +
                white    0.1941    0.0607    3.2002    0.0014  **
                 in50   -0.1815    0.

### Power Outage — Picking up takeout  (`Pick up`)

Dependent variable: `ext_powerout_takeout_pickup`  |  14 terms  |  N = 2,300

In [67]:
# Power Outage - Picking up takeout
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_takeout_pickup'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'CR',
    'female',
    'age_3150',
    'non_wrkr',
    'white',
    'asian',
    'in50',
    'in50100',
    'NetworkDensity_medium',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Picking up takeout   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Pick up')] = tbl
except NameError: pass

Power Outage - Picking up takeout   dv=ext_powerout_takeout_pickup   (3 categories)
N = 2381   logL = -2364.47   AIC = 4760.95   pseudo-R2 = 0.0154
thresholds strictly increasing: True   ->  [-0.8901  0.6149]
------------------------------------------------------------------------------
             Variable      Coef        SE         z         p sig
       powerout_imp_2   -0.0444    0.0634   -0.7012    0.4832    
       powerout_imp_3    0.0441    0.0649    0.6792    0.4970    
       powerout_imp_4    0.1578    0.0824    1.9148    0.0555   +
       powerout_imp_5    0.0930    0.1034    0.9002    0.3680    
                   CR    0.0476    0.0277    1.7206    0.0853   +
               female   -0.1243    0.0479   -2.5951    0.0095  **
             age_3150    0.0878    0.0532    1.6502    0.0989   +
             non_wrkr   -0.1243    0.0507   -2.4498    0.0143   *
                white    0.1140    0.0643    1.7722    0.0764   +
                asian    0.3219    0.1429    2.2519 

### Power Outage — Having food delivered  (`Delivery`)

Dependent variable: `ext_powerout_food_delivery`  |  12 terms  |  N = 2,300

In [68]:
# Power Outage - Having food delivered
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/powerout_data.csv', low_memory=False)
dv = 'ext_powerout_food_delivery'

x_vars = [
    'powerout_imp_2',
    'powerout_imp_3',
    'powerout_imp_4',
    'powerout_imp_5',
    'SE',
    'PR',
    'age_5165',
    'in50',
    'in50100',
    'hhsize1',
    'hhsize2',
    'NatWalkInd_low',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Power Outage - Having food delivered   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('powerout', 'Delivery')] = tbl
except NameError: pass

Power Outage - Having food delivered   dv=ext_powerout_food_delivery   (3 categories)
N = 2381   logL = -2253.92   AIC = 4535.84   pseudo-R2 = 0.0110
thresholds strictly increasing: True   ->  [-0.9477  0.7307]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
powerout_imp_2   -0.0232    0.0641   -0.3619    0.7174    
powerout_imp_3    0.1053    0.0656    1.6060    0.1083    
powerout_imp_4    0.0657    0.0827    0.7949    0.4267    
powerout_imp_5    0.1816    0.1040    1.7462    0.0808   +
            SE    0.0656    0.0296    2.2173    0.0266   *
            PR   -0.0450    0.0295   -1.5255    0.1271    
      age_5165   -0.1259    0.0542   -2.3226    0.0202   *
          in50   -0.1372    0.0629   -2.1793    0.0293   *
       in50100   -0.1732    0.0648   -2.6713    0.0076  **
       hhsize1   -0.0980    0.0608   -1.6126    0.1068    
       hhsize2   -0.1562    0.0553   -2.8235    0.0048  **
Na

---

# Earthquakes

`event_data/earthquake_data.csv` — 5 models.

### Earthquakes — Go about business as usual  (`Usual`)

Dependent variable: `ext_earthquake_lkly_normal_business`  |  11 terms  |  N = 611

In [69]:
# Earthquakes - Go about business as usual
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/earthquake_data.csv', low_memory=False)
dv = 'ext_earthquake_lkly_normal_business'

x_vars = [
    'earthquake_imp_2',
    'earthquake_imp_3',
    'earthquake_imp_4',
    'earthquake_imp_5',
    'PR',
    'female',
    'age_5165',
    'age_65p',
    'white',
    'tcom_no',
    'rural',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Earthquakes - Go about business as usual   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('earthquake', 'Usual')] = tbl
except NameError: pass

Earthquakes - Go about business as usual   dv=ext_earthquake_lkly_normal_business   (5 categories)
N = 611   logL = -914.81   AIC = 1859.62   pseudo-R2 = 0.0358
thresholds strictly increasing: True   ->  [-1.7272 -1.0306 -0.3369  0.6184]
------------------------------------------------------------------------------
        Variable      Coef        SE         z         p sig
earthquake_imp_2   -0.2676    0.1173   -2.2809    0.0226   *
earthquake_imp_3   -0.3532    0.1161   -3.0417    0.0024  **
earthquake_imp_4   -0.1865    0.1409   -1.3233    0.1857    
earthquake_imp_5   -0.0551    0.2003   -0.2751    0.7832    
              PR    0.1755    0.0550    3.1939    0.0014  **
          female   -0.2742    0.0892   -3.0731    0.0021  **
        age_5165   -0.4379    0.1159   -3.7778    0.0002 ***
         age_65p   -0.4832    0.1079   -4.4782    0.0000 ***
           white    0.1856    0.1040    1.7841    0.0744   +
         tcom_no   -0.2016    0.1134   -1.7783    0.0753   +
           r

### Earthquakes — Staying at home  (`Home`)

Dependent variable: `ext_earthquake_stay_home`  |  8 terms  |  N = 182

> ⚠ **`ac_c` cuts this sample from 611 to 182.** The published model used raw `ac` (with `-9` treated as a value). Swap the commented line below to reproduce it.

In [70]:
# Earthquakes - Staying at home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/earthquake_data.csv', low_memory=False)
dv = 'ext_earthquake_stay_home'

x_vars = [
    'earthquake_imp_2',
    'earthquake_imp_3',
    'earthquake_imp_4',
    'earthquake_imp_5',
    'non_wrkr',
    'white',
    'asian',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['earthquake_imp_2', 'earthquake_imp_3', 'earthquake_imp_4', 'earthquake_imp_5', 'non_wrkr', 'white', 'asian', 'ac_raw']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Earthquakes - Staying at home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('earthquake', 'Home')] = tbl
except NameError: pass

Earthquakes - Staying at home   dv=ext_earthquake_stay_home   (3 categories)
N = 611   logL = -544.26   AIC = 1106.52   pseudo-R2 = 0.0175
thresholds strictly increasing: True   ->  [-1.0012  0.629 ]
------------------------------------------------------------------------------
        Variable      Coef        SE         z         p sig
earthquake_imp_2    0.2862    0.1286    2.2246    0.0261   *
earthquake_imp_3    0.2755    0.1272    2.1667    0.0303   *
earthquake_imp_4    0.2669    0.1523    1.7519    0.0798   +
earthquake_imp_5    0.3580    0.2171    1.6487    0.0992   +
        non_wrkr    0.2219    0.0999    2.2221    0.0263   *
           white    0.1894    0.1194    1.5868    0.1126    
           asian    0.5653    0.2280    2.4800    0.0131   *
     threshold_1   -1.0012       NaN       NaN       NaN    
     threshold_2    0.6290       NaN       NaN       NaN    
------------------------------------------------------------------------------ 
sig: *** p<0.001  ** p<0.01  * 

### Earthquakes — Using a car for traveling  (`Car`)

Dependent variable: `ext_earthquake_car_travel`  |  9 terms  |  N = 611

In [71]:
# Earthquakes - Using a car for traveling
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/earthquake_data.csv', low_memory=False)
dv = 'ext_earthquake_car_travel'

x_vars = [
    'earthquake_imp_2',
    'earthquake_imp_3',
    'earthquake_imp_4',
    'earthquake_imp_5',
    'PR',
    'female',
    'age_5165',
    'age_65p',
    'hhsize1',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Earthquakes - Using a car for traveling   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('earthquake', 'Car')] = tbl
except NameError: pass

Earthquakes - Using a car for traveling   dv=ext_earthquake_car_travel   (3 categories)
N = 611   logL = -580.95   AIC = 1183.90   pseudo-R2 = 0.0479
thresholds strictly increasing: True   ->  [-0.7755  0.7939]
------------------------------------------------------------------------------
        Variable      Coef        SE         z         p sig
earthquake_imp_2   -0.2130    0.1270   -1.6773    0.0935   +
earthquake_imp_3    0.0781    0.1242    0.6289    0.5294    
earthquake_imp_4    0.3310    0.1500    2.2062    0.0274   *
earthquake_imp_5    0.2036    0.2097    0.9708    0.3317    
              PR    0.1172    0.0588    1.9927    0.0463   *
          female   -0.2220    0.0957   -2.3205    0.0203   *
        age_5165   -0.3741    0.1245   -3.0057    0.0026  **
         age_65p   -0.4970    0.1137   -4.3699    0.0000 ***
         hhsize1    0.2225    0.1089    2.0434    0.0410   *
     threshold_1   -0.7755       NaN       NaN       NaN    
     threshold_2    0.7939       NaN   

### Earthquakes — Taking public transit  (`Transit`)

Dependent variable: `ext_earthquake_transit_use`  |  11 terms  |  N = 611

> Note: uses cleaned `hispanic_c` (1 = Hispanic). The published model used raw `hispanic` (1 = Yes, 2 = No), so this coefficient's **sign is reversed** relative to the published table.

In [72]:
# Earthquakes - Taking public transit
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/earthquake_data.csv', low_memory=False)
dv = 'ext_earthquake_transit_use'

x_vars = [
    'earthquake_imp_2',
    'earthquake_imp_3',
    'earthquake_imp_4',
    'earthquake_imp_5',
    'SE',
    'female',
    'non_wrkr',
    'bs_grad',
    'hispanic_c',
    'hhsize1',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Earthquakes - Taking public transit   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('earthquake', 'Transit')] = tbl
except NameError: pass

Earthquakes - Taking public transit   dv=ext_earthquake_transit_use   (3 categories)
N = 611   logL = -555.61   AIC = 1137.22   pseudo-R2 = 0.0335
thresholds strictly increasing: True   ->  [-0.8817  0.8977]
------------------------------------------------------------------------------
        Variable      Coef        SE         z         p sig
earthquake_imp_2    0.0729    0.1289    0.5653    0.5718    
earthquake_imp_3    0.2203    0.1278    1.7238    0.0847   +
earthquake_imp_4    0.3517    0.1576    2.2318    0.0256   *
earthquake_imp_5    0.2169    0.2186    0.9920    0.3212    
              SE   -0.1264    0.0650   -1.9462    0.0516   +
          female   -0.1980    0.0991   -1.9980    0.0457   *
        non_wrkr   -0.2710    0.1174   -2.3075    0.0210   *
         bs_grad   -0.2074    0.0988   -2.0990    0.0358   *
      hispanic_c    0.3708    0.1494    2.4826    0.0130   *
         hhsize1    0.2973    0.1098    2.7082    0.0068  **
         tcom_no   -0.3218    0.1349   -2.

### Earthquakes — Working from home  (`WFH`)

Dependent variable: `ext_earthquake_wfh`  |  5 terms  |  N = 343  |  workers only (`empsta < 3`)

In [73]:
# Earthquakes - Working from home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/earthquake_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_earthquake_wfh'

x_vars = [
    'earthquake_imp_2',
    'earthquake_imp_3',
    'earthquake_imp_4',
    'earthquake_imp_5',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Earthquakes - Working from home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('earthquake', 'WFH')] = tbl
except NameError: pass

Earthquakes - Working from home   dv=ext_earthquake_wfh   (3 categories)
N = 343   logL = -300.41   AIC = 614.83   pseudo-R2 = 0.0299
thresholds strictly increasing: True   ->  [-1.3108  0.5445]
------------------------------------------------------------------------------
        Variable      Coef        SE         z         p sig
earthquake_imp_2    0.0209    0.1867    0.1117    0.9111    
earthquake_imp_3    0.3649    0.1736    2.1015    0.0356   *
earthquake_imp_4    0.1826    0.2012    0.9079    0.3639    
earthquake_imp_5    0.0468    0.2480    0.1888    0.8503    
         tcom_no   -0.4823    0.1373   -3.5115    0.0004 ***
     threshold_1   -1.3108       NaN       NaN       NaN    
     threshold_2    0.5445       NaN       NaN       NaN    
------------------------------------------------------------------------------ 
sig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10


---

# Flooding

`event_data/flooding_data.csv` — 5 models.

### Flooding — Go about business as usual  (`Usual`)

Dependent variable: `ext_flooding_lkly_normal_business`  |  11 terms  |  N = 1,077

In [74]:
# Flooding - Go about business as usual
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/flooding_data.csv', low_memory=False)
dv = 'ext_flooding_lkly_normal_business'

x_vars = [
    'flooding_imp_2',
    'flooding_imp_3',
    'flooding_imp_4',
    'flooding_imp_5',
    'CR',
    'PR',
    'female',
    'age_3150',
    'non_wrkr',
    'white',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Flooding - Go about business as usual   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('flooding', 'Usual')] = tbl
except NameError: pass

Flooding - Go about business as usual   dv=ext_flooding_lkly_normal_business   (5 categories)
N = 1077   logL = -1627.97   AIC = 3285.95   pseudo-R2 = 0.0273
thresholds strictly increasing: True   ->  [-1.6791 -0.9863 -0.3223  0.6471]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
flooding_imp_2   -0.4763    0.0870   -5.4770    0.0000 ***
flooding_imp_3   -0.5373    0.0916   -5.8686    0.0000 ***
flooding_imp_4   -0.6637    0.1157   -5.7348    0.0000 ***
flooding_imp_5   -0.6166    0.1425   -4.3278    0.0000 ***
            CR    0.0853    0.0394    2.1667    0.0303   *
            PR    0.1092    0.0404    2.7022    0.0069  **
        female   -0.1754    0.0665   -2.6380    0.0083  **
      age_3150    0.2043    0.0695    2.9419    0.0033  **
      non_wrkr   -0.1664    0.0800   -2.0809    0.0374   *
         white    0.1801    0.0809    2.2253    0.0261   *
       tcom_no   -0.3384    0.0874   

### Flooding — Staying at home  (`Home`)

Dependent variable: `ext_flooding_stay_home`  |  8 terms  |  N = 1,041

In [75]:
# Flooding - Staying at home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/flooding_data.csv', low_memory=False)
dv = 'ext_flooding_stay_home'

x_vars = [
    'flooding_imp_2',
    'flooding_imp_3',
    'flooding_imp_4',
    'flooding_imp_5',
    'CR',
    'female',
    'bs_grad',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Flooding - Staying at home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('flooding', 'Home')] = tbl
except NameError: pass

Flooding - Staying at home   dv=ext_flooding_stay_home   (3 categories)
N = 1077   logL = -928.58   AIC = 1877.15   pseudo-R2 = 0.0170
thresholds strictly increasing: True   ->  [-1.26    0.3784]
------------------------------------------------------------------------------
          Variable      Coef        SE         z         p sig
    flooding_imp_2    0.2181    0.0954    2.2876    0.0222   *
    flooding_imp_3    0.2013    0.0992    2.0300    0.0424   *
    flooding_imp_4    0.0884    0.1252    0.7066    0.4798    
    flooding_imp_5    0.2529    0.1560    1.6216    0.1049    
                CR    0.1180    0.0427    2.7666    0.0057  **
            female    0.1691    0.0731    2.3137    0.0207   *
           bs_grad    0.1945    0.0783    2.4855    0.0129   *
TransitAccess_high    0.2095    0.1073    1.9529    0.0508   +
       threshold_1   -1.2600       NaN       NaN       NaN    
       threshold_2    0.3784       NaN       NaN       NaN    
--------------------------------

### Flooding — Using a car for traveling  (`Car`)

Dependent variable: `ext_flooding_car_travel`  |  11 terms  |  N = 1,041

In [76]:
# Flooding - Using a car for traveling
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/flooding_data.csv', low_memory=False)
dv = 'ext_flooding_car_travel'

x_vars = [
    'flooding_imp_2',
    'flooding_imp_3',
    'flooding_imp_4',
    'flooding_imp_5',
    'SE',
    'PR',
    'age_65p',
    'child',
    'work_out',
    'hhveh0',
    'TransitAccess_high',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Flooding - Using a car for traveling   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('flooding', 'Car')] = tbl
except NameError: pass

Flooding - Using a car for traveling   dv=ext_flooding_car_travel   (3 categories)
N = 1077   logL = -1011.67   AIC = 2049.34   pseudo-R2 = 0.0344
thresholds strictly increasing: True   ->  [-0.237   1.2639]
------------------------------------------------------------------------------
          Variable      Coef        SE         z         p sig
    flooding_imp_2   -0.0647    0.0938   -0.6903    0.4900    
    flooding_imp_3   -0.0335    0.0992   -0.3377    0.7356    
    flooding_imp_4    0.0755    0.1242    0.6079    0.5433    
    flooding_imp_5    0.2651    0.1506    1.7604    0.0783   +
                SE    0.1417    0.0445    3.1801    0.0015  **
                PR   -0.1171    0.0430   -2.7226    0.0065  **
           age_65p   -0.2659    0.0897   -2.9643    0.0030  **
             child    0.1282    0.0827    1.5511    0.1209    
          work_out    0.6082    0.1507    4.0354    0.0001 ***
            hhveh0    0.2320    0.1057    2.1949    0.0282   *
TransitAccess_high  

### Flooding — Taking public transit  (`Transit`)

Dependent variable: `ext_flooding_transit_use`  |  10 terms  |  N = 323

> ⚠ **`ac_c` cuts this sample from 1,041 to 323.** The published model used raw `ac` (with `-9` treated as a value). Swap the commented line below to reproduce it.

In [77]:
# Flooding - Taking public transit
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/flooding_data.csv', low_memory=False)
dv = 'ext_flooding_transit_use'

x_vars = [
    'flooding_imp_2',
    'flooding_imp_3',
    'flooding_imp_4',
    'flooding_imp_5',
    'CR',
    'age_3150',
    'edu_bs',
    'child',
    'ac_c',
    'NatWalkInd_very_high',
]

# To reproduce the published model instead, use raw ac (-9 kept as a value):
# x_vars = ['flooding_imp_2', 'flooding_imp_3', 'flooding_imp_4', 'flooding_imp_5', 'CR', 'age_3150', 'edu_bs', 'child', 'ac_raw', 'NatWalkInd_very_high']

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Flooding - Taking public transit   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('flooding', 'Transit')] = tbl
except NameError: pass

Flooding - Taking public transit   dv=ext_flooding_transit_use   (3 categories)
N = 1077   logL = -944.80   AIC = 1913.59   pseudo-R2 = 0.0181
thresholds strictly increasing: True   ->  [-0.5963  1.2966]
------------------------------------------------------------------------------
            Variable      Coef        SE         z         p sig
      flooding_imp_2   -0.0584    0.0947   -0.6171    0.5372    
      flooding_imp_3    0.0006    0.0994    0.0058    0.9954    
      flooding_imp_4    0.0872    0.1250    0.6975    0.4855    
      flooding_imp_5   -0.0230    0.1524   -0.1506    0.8803    
                  CR    0.0788    0.0419    1.8792    0.0602   +
            age_3150   -0.1420    0.0779   -1.8239    0.0682   +
              edu_bs    0.2542    0.0878    2.8953    0.0038  **
               child    0.1635    0.0818    1.9997    0.0455   *
                ac_c    0.1537    0.0787    1.9520    0.0509   +
NatWalkInd_very_high    0.2291    0.0791    2.8954    0.0038  **
  

### Flooding — Working from home  (`WFH`)

Dependent variable: `ext_flooding_wfh`  |  7 terms  |  N = 585  |  workers only (`empsta < 3`)

In [78]:
# Flooding - Working from home
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings; warnings.filterwarnings('ignore')

d  = pd.read_csv('./event_data/flooding_data.csv', low_memory=False)
d = d[d['empsta'] < 3]                      # workers only
dv = 'ext_flooding_wfh'

x_vars = [
    'flooding_imp_2',
    'flooding_imp_3',
    'flooding_imp_4',
    'flooding_imp_5',
    'CR',
    'bs_grad',
    'tcom_no',
]

md_ = d[x_vars + [dv]].dropna()
res = OrderedModel(md_[dv], md_[x_vars], distr='probit').fit(method='bfgs', disp=False, maxiter=300)

k    = res.model.k_levels - 1
thr  = res.model.transform_threshold_params(res.params.values)[1:-1]   # real cutpoints, R-comparable
assert np.all(np.diff(thr) > 0), f'thresholds not strictly increasing: {thr}'

p    = res.pvalues[:-k].values
star = ['***' if x<.001 else '**' if x<.01 else '*' if x<.05 else '+' if x<.1 else '' for x in p]
tbl  = pd.DataFrame({'Variable': x_vars, 'Coef': res.params[:-k].values, 'SE': res.bse[:-k].values,
                     'z': res.tvalues[:-k].values, 'p': p, 'sig': star})
tbl  = pd.concat([tbl, pd.DataFrame({'Variable': [f'threshold_{i+1}' for i in range(len(thr))],
                     'Coef': thr, 'SE': np.nan, 'z': np.nan, 'p': np.nan, 'sig': ''})], ignore_index=True)
tbl.attrs.update(nobs=int(res.nobs), llf=res.llf, aic=res.aic, prsquared=res.prsquared)

print(f'Flooding - Working from home   dv={dv}   ({md_[dv].nunique()} categories)')
print(f'N = {int(res.nobs)}   logL = {res.llf:.2f}   AIC = {res.aic:.2f}   pseudo-R2 = {res.prsquared:.4f}')
print(f'thresholds strictly increasing: {bool(np.all(np.diff(thr) > 0))}   ->  {np.round(thr, 4)}')
print('-'*78)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('-'*78, '\nsig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10')

try: MODELS[('flooding', 'WFH')] = tbl
except NameError: pass

Flooding - Working from home   dv=ext_flooding_wfh   (3 categories)
N = 585   logL = -503.43   AIC = 1024.86   pseudo-R2 = 0.0237
thresholds strictly increasing: True   ->  [-1.1567  0.7498]
------------------------------------------------------------------------------
      Variable      Coef        SE         z         p sig
flooding_imp_2    0.1477    0.1388    1.0643    0.2872    
flooding_imp_3    0.1848    0.1401    1.3197    0.1869    
flooding_imp_4    0.1594    0.1691    0.9421    0.3461    
flooding_imp_5    0.5211    0.2136    2.4399    0.0147   *
            CR    0.1308    0.0592    2.2083    0.0272   *
       bs_grad    0.2055    0.1020    2.0144    0.0440   *
       tcom_no   -0.1497    0.1004   -1.4916    0.1358    
   threshold_1   -1.1567       NaN       NaN       NaN    
   threshold_2    0.7498       NaN       NaN       NaN    
------------------------------------------------------------------------------ 
sig: *** p<0.001  ** p<0.01  * p<0.05  + p<0.10


---

# Summary — build `model_coefficients_all_events.csv`

Run after the model cells above (needs the `MODELS = {}` cell to have been run first).

In [79]:
EXPECTED = {
 'heat':       ['Usual','Home','Car','Transit','WFH','WFO','Dine in','Pick up','Delivery'],
 'cold':       ['Usual','Home','Car','Transit','WFH','WFO','Dine in','Pick up','Delivery'],
 'powerout':   ['Usual','Home','Car','Transit','Dine in','Pick up','Delivery'],
 'earthquake': ['Usual','Home','Car','Transit','WFH'],
 'flooding':   ['Usual','Home','Car','Transit','WFH'],
}
expected_keys = [(e, a) for e, acts in EXPECTED.items() for a in acts]
missing = [k for k in expected_keys if k not in MODELS]
print(f'{len(MODELS)} of {len(expected_keys)} models estimated')
if missing:
    print(f'NOT YET RUN ({len(missing)}):')
    for e, a in missing: print(f'   {e} / {a}')
else:
    print('all models present')

35 of 35 models estimated
all models present


In [81]:
import pandas as pd
rows = []
for (ev, act), tbl in MODELS.items():
    for _, r in tbl.iterrows():
        v = str(r['Variable'])
        rows.append(dict(event=ev, activity=act,
                         term='threshold' if v.startswith('threshold_') else 'coef',
                         variable=v, value=float(r['Coef']),
                         se=None if pd.isna(r['SE']) else float(r['SE']),
                         p=None if pd.isna(r['p']) else float(r['p']),
                         n=int(tbl.attrs.get('nobs', 0))))
coef_all = pd.DataFrame(rows).sort_values(['event','activity','term','variable']).reset_index(drop=True)
coef_all.to_csv('./model_coefficients_all_events.csv', index=False)
print('wrote model_coefficients_all_events.csv', coef_all.shape)
print(coef_all.groupby('event').agg(models=('activity','nunique'), terms=('variable','size')))
coef_all.head(12)

wrote model_coefficients_all_events.csv (479, 8)
            models  terms
event                    
cold             9    124
earthquake       5     55
flooding         5     59
heat             9    133
powerout         7    108


,event,activity,term,variable,value,se,p,n
0,cold,Car,coef,PopDens_high,0.153328,0.051488,0.002902,2434
1,cold,Car,coef,SE,0.082342,0.028912,0.004398,2434
2,cold,Car,coef,age_5165,-0.130903,0.061482,0.033242,2434
3,cold,Car,coef,age_65p,-0.148588,0.066471,0.025392,2434
4,cold,Car,coef,black,0.208190,0.069282,0.002656,2434
5,cold,Car,coef,cold_imp_2,-0.059520,0.067781,0.379882,2434
6,cold,Car,coef,cold_imp_3,-0.056000,0.069654,0.421414,2434
7,cold,Car,coef,cold_imp_4,0.021267,0.083959,0.800031,2434
8,cold,Car,coef,cold_imp_5,0.187895,0.107099,0.079361,2434
9,cold,Car,coef,female,-0.082068,0.048777,0.092469,2434


### Threshold monotonicity — all models at once

Each cell already asserts this, but this re-checks everything collected in `MODELS`.

In [82]:
import numpy as np
bad = []
for (ev, act), tbl in MODELS.items():
    thr = tbl[tbl.Variable.str.startswith('threshold_')].Coef.values
    if not np.all(np.diff(thr) > 0):
        bad.append((ev, act, thr))
print(f'checked {len(MODELS)} models')
print('all thresholds strictly increasing' if not bad else f'PROBLEM in {len(bad)}: {bad}')

checked 35 models
all thresholds strictly increasing
